# Databricks data ingestion
There are three methods that could be used to ingest data from files stored in the cloud storage.
- CREATE TABLE AS (CTAS) : 
    - It creates a delta table by default from files stored in the cloud object storage.
    - The ```read_files()``` function is used to read files from a specified loaction and return the data in a tabular format.
    - It offers several capabilities:
        - Supports various file formats like JSON, csv, xml, text, binaryfilem paraquet, avro and orc
        - Automatically detects file format and infers a unified schema across all files.
        - Allows you to specify format-specific options for greater control when reading source files.
        - Can be used in streaming tables to incrementally ingest files into delta lake using auto loader. We will learn more about auto loader shortly. 
- COPY INTO : 
    - This is used to copy files from cloud storage into the delta table. 
    - This command performs bulk load from files in cloud object storage into the table, and in this example, it will load files into the empty table new_table.
    - The FROM clause specifies the location of the csv files
    - You start by creating a table which can be defined with or without a schema. In this case, we will create a table named new_table without a schema.
    - COPY INTO is ideal for situations where the cloud storage location is continuously adding files, since it is a reltriable and indepodent operation designed for incremental batch ingestion.
        - What that means is : COPY INTO will skip any files that have already been loaded into the table, and only new files will be ingested. 
        - Now lets go over some of the key aspects of COPY INTO: 
            - It supports various common file types liek parquet, JSON, XML and others.
            - The FROM clause specifies the path of the cloud storage where new files are being continously added.
            - FORMAT_OPTIONS{} controls how the source files are parsed and interpreted and the available options will depend on the file format you are working with.
            - COPY_OPTIONS() lets you control the behaviour of COPY INTO operation itself. For example: options like schema evolution using mergeSchema, or idempotency using force.
- Auto Loader : 
    - Incrementally and efficiently process new data files as they arrive in cloud storage without any additional setup.
    - Auto loader has support for both python and sql (leveraging declarative pipelines)
    - You can use Auto Loader to process billions of files. 
    - Auto loader is built upon **Spark Structured Streaming**

## Schema : 
- A schema (in terms of databases) is a formal language which describes the structure of data (blueprint) of a database.
- A schema can define many different data structures that serve different purpose for a database.
- Different data structures (relational databases):
    - Tables
    - Fields
    - Views
    - Relationships
    - Indexes
    - Packages
    - Procedures
    - Functions
    - XML schemas
    - Queues
    - Triggers
    - Types
    - Sequences
    - materialized views
    - Synonyms
    - database links
    - Directories
## Schemaless : 
- Schemaless is when the primary "cell" of database can accept many types.
- This allows developers to forgo the upfront data modelling.
- Common schemaless databases are  : 
    - Key/Value
    - Document
    - Columns
        - Wide column
    - graph

## Data Documents
- A data document defines the collective form in which data exists.
- Common types of data documents : 
    - Datasets : a logical grouping of data
    - Databases : structured data that can be quickly accessed and searched
    - Datastores : unstructured or semi-structured data to housing data
    - Data warehouse : structured or semi-structured data for creating reports and analytics.
    - Notebooks : data that is arranged in pages, designed for easy consumption

## Data sets
- A data sets is a logical grouping of units of data that generally are closely related and/or share the same data structure.
- Just because I said data structure doesn't always mean that the data itself is structured, it can be a semi-structured or un-structured data
- There are publically available data sets that are used in the learning of statistics, data analytics, machine learning
- MNIST database Images of handwritten digits used to test classification, clustering and image processing algorithms.
- Commonly used when learning how to build computer vision ML models to translate handwriting into digital text.
- COCO dataset (Common objects in Context dataset) : A dataset which contains many common images using a JSON file (coco format) that identify objects or segments within an image.
- IMDB riviews datasets : A movie dataset with 25,000 highly popular movie reviews for training and 25000 for testing.


## Query and Querying 
- A query is a request for data results (reads) or to perform operations such as inserting updating deleting data (writes).
- A query can perform maintanance operations on the data and is not always restricted to just working with the data that resides the database.
- Querying : This is an act of performing a query
- what is a query language ? : A scripting language designed as the format to submit a request or actions to the database. Notable query languages: 
    - SQL
    - GraphSQL
    - Kusto
    - Xpath
    - Gremlin

## Batch vs Stream processing
### Batch processing
![batch_processing](images/batch_processing.png)
- When you send batches (a collection) of data to be processed. 
- Batches are generally scheduled example : Every day at 1PM
- Batches are not real-time.
- Batche processing is ideal for very large processing workloads.
- Batch processing is more cost-effective
### Stream processing
![stream_processing](images/stream_processing.png)
When you process data as soon as it arrives : 
- Produces will send data to a stream 
- Consumers will pull from the stream
- A data can be held in a stream for a period of time so that we can have a better re-usability of data.
- It is suitable for real-time processing (streaming-video)
- Much more expensive than batch processing.


## Pivot table
- A pivot table is a table of statistics that summarizes the data of a more extensive table from a : Database, Spreadsheet or Business intelligence (BI) tool
- Pivot tables are a technique in data processing.
- They arrange and rearrange (or "Pivot") statistics in order to draw attention to useful information
- This leads to finding figures and facts quickly making them integral to data analysis.
- Here is the example table :

    | Region | Product | Sales |
    | ------ | ------- | ----- |
    | East   | Apples  | 100   |
    | East   | Bananas | 150   |
    | West   | Apples  | 200   |
    | West   | Bananas | 120   |

- Here is the pivot table of the above table : 

    | Region    | Apples  | Bananas | Total   |
    | --------- | ------- | ------- | ------- |
    | East      | 100     | 150     | 250     |
    | West      | 200     | 120     | 320     |
    | **Total** | **300** | **270** | **570** |

- Here is a sample code for generating a pivot table from a table (python > Pandas)
    ```python
    import pandas as pd

    data = {
        "Region": ["East", "East", "West", "West"],
        "Product": ["Apples", "Bananas", "Apples", "Bananas"],
        "Sales": [100, 150, 200, 120],
    }
    df = pd.DataFrame(data)

    pivot = df.pivot_table(index="Region", columns="Product", values="Sales", aggfunc="sum")
    print(pivot)
    ```
- Here is a sample code for generating a pivot table from a table in (pyspark)
    ```python
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import sum

    spark = SparkSession.builder.getOrCreate()

    data = [("East", "Apples", 100),
            ("East", "Bananas", 150),
            ("West", "Apples", 200),
            ("West", "Bananas", 120)]

    df = spark.createDataFrame(data, ["Region", "Product", "Sales"])

    pivot_df = df.groupBy("Region").pivot("Product").agg(sum("Sales"))
    pivot_df.show()
    ```
- **When to use pivot table:**
    - Summarize large datasets quickly
    - Find totals, averages or counts grouped by categories.
    - compare data across multiple dimensions (example region vs products)
    - Explore patterns or trends in your dataset.
    


## Relational data 
### Tables : 
- A logical grouping of rows and columns. Think like a Excel spreadsheet.
- Tabular data --- data that makes use of table data strucutres
### Views : 
- Views is a result set of a stored query on data stored in memory (a temporary or virtual table)
### Materialized Views : 
- Material Views is a result set of stored query on data stored on disk.
### Indexes : 
- A copy of your data sorted by one or multiple columns for faster reads at cost of storage.
### Constraints : 
- Rules applied to writes, that can ensure data integrity example : don't allow duplicate records.
### Triggers : 
- A function that is triggered on specific database events.
### Primary key : 
- One or multiple columns that uniquely identify a table in a row
### Foreign key :
- A column which holds the value of primary key from another key to establish a relationship.
    - A relationship is when two tables have a reference to one another to join data togeather.

## Relational data -- Relationships
- A relational database establish connection to other tables via foreign keys referencing another table's primary key.
- Types of relation : 
    - **One to one** : A country has a capital
    - **One to many** : A Store has many customers
    - **Many to many** : A project has many tasks and Tasks can belong to many projects.
    - **Many to Many (via Join/Junction table)** : A student has many classes through enrollments. A class has many students through enrollments.
    

## Indexes
- A database index is a data structure that improves the speed of reads from the database table by storing the same or partial redundant data organized in a more efficient logical order.
- A logical order is commonly determined by one or more sort ke(s)
- A common data structure of an index is a balanced Tree (B-Tree)

## Non-relational data
- A non-relational database stores data in a non-tabular form and will be optimized for different kinds of data-strucutres.
- Types of non-relational databases:
    - **Key/value**
        - Each value has a key
        - Designed to scale
        - Only simple lookups
    - **Document**
        - Primary entity is a JSON-like data-structure called a document.
    - **Columnar**
        - Has a table-like structure but data is stored around columns instead of rows.
    - **GRAPH**
        - Data is represented with nodes and structures. Where relationships matter
        

## Data integrity and Data corruption
### Data integrity
- Data integrity is the maintenance and assurance of data accuracy and consitency over its entire life-cycle.
- Its used as proxy term for data quality, data validation is a pre-requisite for data integrity.
- The goal of data integrity ensure data is recorded exactly as intended.
### Data corruption
- Data corruption is the act or state of data not being in the intended state and will result in data or misinformation.
- Data corruption occurs when unintended changes result when reading and writing:
    - Unexpected hardware failure.
    - Human error when inputing or modifying data
    - Malicious actors with intent of corrupting your data.
    - Unforeseen side effects for automated operations via computer code.
### Ways to insure data integrity:
- Have a well defined and documented data modelling.
- Logical constraints on your database items.
- Redundant and versions of your data to compare and restore
- Human analysis of the data
- Hash functions to determine if changes have been tampered

## Normalized vs De-normalized data
### Normalized 
A schema design to store non-redundant and consistent data.
- Data integrity is maintained
- Little to no redundant data
- Many tables
- Optimizes for storage of data
### Denormalized
A schema that combines data so that accessing data (querying) is fast.
- data integrity is not maintained
- Redundant data is common
- Fewer tables
- Excessive data, storage is less optimal

## Strongly consistent vs Eventually consistent
### **What is data consistency?**
When data being kept in two different place and whether the data exactly match or do not match.

When you have to have duplicate your data in many places and need to keep them up to date to be exact matching, based on how data is transmitted and service levels cloud service

#### Strongly consistent
Every time you request data (query) you can expect consistent data to be returned with x time (1 seconds)

We will never return to you with old data. But you have to wait at least 2 seconds for the query to return.

#### Eventually consistent
When you request data you may get back inconsistent data within 2 seconds.

We are giving you whatever data is currently in the database you may get new data or old data but if you wait a little bit longer it will generally be up to date.

## Synchronous vs Asynchronous
### Synchronous 
Continuous stream of data that is synchronized by a timer or clock (guarantee of time). Can only access data once transfer is complete.
- Guarantee consitency of data return at time of access
- Slower access times
### Asynchronous
Continuous stream of data seperated by start and stop bits (no guarantee of time)
Can access data anytime but may return older version or empty
placeholder
- Faster access times not guarantee of consistency


## Data source 
A data source is where data originates from. An analytics tool may be connected to various data sources to create a visualization or report.

A data source could be a : 
- Data Lake 
- Data Warehouse
- Datastore
- Database
- Data requested on demand from an API endpoint from a web app
- Flat files (example excel spreadsheet)
## Data Store
- A data store is a repository for persistently storing and managing collections of unstructured or semi-structured data.
- A database is a sub-set of a data store.
- It is generally a data store indicates working unstructured or semi-structured data.
- A datastore can be specialized in storing : 
    - Relational databases
    - NoSQL databases
    - Object oriented databases
- Data stores are designed to be distributed across many machines.
- Directory service
## Database 
- A database is a data-store that stores semi-structred and structured data.
- A database is more complex data stores because it requires using formal design and modeling technoques.
- Database can be generally categorized as either:
    - Relational databases : 
        - Structured data that strongly represents tabular data (tables, rows and columns) 
        - Row oriented or columnar oriented
    - Non-relational databases : 
        - Semi-structured that may or may not distantly resemble tabular data.
- Databases have a rich set of functionality
    - specialized language to query (retrieve data) 
    - specialized modeling stratagies to optimize retrieval for different use cases
    - more fine tune control over the transformation of the data into useful data structures or reports
## Data warehouse 
- A relational datastore designed for analytics workloads, which generally column-oriented data-store
- Companies will have terabytes of rows of data and they need a fast way to be able to produce analytics reports.
- Data warehouses generally perform aggregation.
- aggregation is grouping data example find the total or average
- data warehouses are optimized around columns since they need to quickly aggregate column data.
- Data warehouses are generally designed to be HOT
- HOT means they can returned queries very very fast even though they have vast amounts of data.
Data warehouses are infrequently accessed meaning they aren't intended for real-time reporting but maybe once or twice a day or once a week to generate business and user reports.
- A data warehouse needs to consume data from a relational databases on a regular basis.
- Generally datawarehouses are read-only where the data is only read and we normally don't use it for transactional data.
## Data Mart
- A data mart subset of a data warehouse
- A data mart will store data under 100 GB and has a single business focus
- Data mart allows different teams or departments to have control over their own dataset for their specific use case.
- Data marts are generally designed to be read-only
- Data marts also increase the frequency at which data can be accessed.
- The cost to query the data is much lower and so queries can be performed multiple times a day or even hourly.
## Data Lake
- A data lake is a centralized storage repository that holds a vast amount of raw data (big data) in either a semi-structured or un-structured format.
- A data Lake lets you all your data without careful design or having to answer questions on the future of the data (Hording for data scientist)
- A data lake is commonly accessed for data workloads such as :
    - Visualizations (Business Intelligence)
    - Real-time analytics 
    - Machine learning
    - on-premise data
- Data lakes are great for data-scientists but its very hard to use data lake for BI reporting
- If data lakes are not well maintained they can become data-swamps (a mess of data or data corruption)
## Data Lakehouse
A data lakehouse combines the best elements of a data lake and a data warehouse.
- Data LakeHouses compared to a Data warehouse can:
    - Support videos, audio and text files
    - Support data science and ML workloads
    - have support for both streaming and ETL
    - Work with many open-source formats
    - Data will generally reside in a data lake or blob stores
- Data lakehouses compared to a data lakes can:
    - perform BI tasks very well
    - much easier to setup and maintain
    - has management features to avoid a data lake becoming data swamp
    - more performant than a data lake
## Data structures
- Data that is organized in a specific storage format that enables easy access and modification.
- A data structure can store various data types
- Data can be abstractly described to have a degree of structure:
    - Unstructured : A bunch of lose data that has no organization or possibly relation
    - Semi-structured : Data that can be browsed or searched with limitations
    - Structured : Data that can be easily browsed or searched.

## Databricks platform
- Databricks is a software company specializing in providing fully managed Apache spark clusters the company founders were the creators of apache spark, Delta Lake and MLFlow.
- Databricks platform : Databricks cloud based spark paltform with an ease to use web UI
    - Launch fully managed spark clusters
    - Launch notebooks to write code and interact with spark
    - Create workspaces to collaborate with team members
    - Role based access controls
    - Create jobs for ETL or data analysis tasks that run immediately or on a schedule
    - Create MLFlow WorkFlows.
    Available on all main cloud service providers example : AWS , Azure and GCP

## DBUs and DSUs
### DBU (Databricks Unit) : 
- It is a unit of compute in databricks LakeHouse.
- Consumption (legacy) cost management service shows usage.
- Databricks dashboards are recommended to use to track your usage.
### DSU (Databricks Storage unit)
- It is a unit of compute for databricks managed storage services
- When creating compute you can define how much DBUs is consumed.
- Example: Serverless starter warehouse (SQL warehouse) created by default is set to 12 DBU / hr.
### Budgets
- Budget is a cost management feature that allows you to set an amount threshold and recieve an email when that budget has been exceeded.
- What can you set:
    - Name of budget
    - Scope based on workspaces
    - Scope based on tags
    - Set a threshold amount in USD
    - Emails to alert when threshold is met

## Medallion architecture
Medallion (AKA Multi-Hop) Architecture is a design pattern to logically organize data in a LakeHouse. Organized into layers - Bronze, Silver and Gold
- Bronze table :
    - Bronze table is raw, ingested directly from source systems in its original form.
    - Ingest raw, often semi-structured or uncleaned data
    - Typically loaded via Auto Loader, Kafka or file formats (JSON, CSV)
    - Used for archiving, reprocessing and validation.
    - Often includes metadata like ingest timestamps
- Silver table : 
    - Silver table data is transformed, filtered and standardized for analytics or modeling.
    - Data is cleaned, de-duplicated and joined
    - Business roles or filtering logic applied.
    - Easier to query and more performant than bronze
    - Often used by analysts, dashboards or ML pipelines.
- Gold Table :
    - gold table data is aggregated or modeled for consumption by business users.
    - Contains KPIs (Key performance indicators), Summaries or metrics.
    - Used by dashboards, reports or stakeholders
    - Often has dimensions and fact tables or BI tools.
    - Optimized for performance and clarity

## Control plane vs Data plane
- Databricks separates the **control** and **data** planes to improve security, scalability and isolation.
- You interact via the control plane while data processing runs in your cloud's data plane.
### Control plane : 
The control plane has : 
- Web UI and Workspace
- Job Scheduling Logic
- Notebooks and APIs
- Orchestrates workflows and configurations
- The control plane is mamanged by databricks
- Control plane runs your interface
### Data plane :
Data plane has :
- Clusters (VMs/containers)
- Spark jobs and code execution
- Secure access to your storage (example, S3, ADLS, GCS)
- It executes actual data operations privately
- The data plane runs on your cloud account
- The data plane runs your data

When we sign up on databricks intelligence platform they had a fully managed one but under the hood they are using some kind of cloud platform it could be either AWS, Azure or GCP

Databricks can be brought to any of the three cloud service provider be it AWS, GCP or Azure, If you use it its gonna spin up the same interface but its gonna be more specific of what its being backed and where your data is being stored 

Databricks can't access your data i.e they are not seeing the end data only you are able to see the end data. They are accessing the data in a secure way.



## Databricks clusters
- A cluster is a collection of virtual machines that run Spark.
- It distributes the work loads across multiple machines using one driver and multiple workers.
- It is used to power notebooks, jobs and pipelines 
- clusters have the ability to auto-scale and can shutdown when idle
### There are two main types of clusters : 
- **All-purpose** : For interactive development
    - An all purpose clusters provides a shared, always-on environment where users can collaboratively develop, test and explore data interactively
    - It supports multiple users at the same time 
    - **How it works?**
        - Databricks provisions virtual machines on the cloud (AWS, Azure or GCP)
        - It installs Apache Spark and configures it for distributed computation
        - Then your notebooks, SQL queries or scripts can run interactively on that cluster 
        - You can attach and detach notebooks dynamically the same clusters can serve multiple users at once.
    - **When to use all-purpose cluster?**
        - Develop and test Pyspark code in notebooks
        - Explore and profile raw data interactively
        - Debug LakeFlow or ETL pipelines before production.
        - Collaborate with teammates on shared analysis
        - Prototype transformations (Bronze -> Silver etc.)
    - You can think of **all-purpose clusters** as your data science workspace whereas job clusters are for automated , production-grade workloads
    - **When not to use it?**
        - Avoid all-purpose clusters in production automation.
        - Instead use:
            - Job clusters : created automatically per job, auto-terminated.
            - Serverless compute : If you are using databricks workFlows 
        - Because: 
            - All-Purpose clusters stay running and can be costly.
            - They can be manually modified reducing reproducibility
            - They are not version-controlled like CI/CD jobs

- **Job** : For dunning automated tasks (like ETL pipelines)
    - A job cluster is a temporary, single-purpose spark cluster that databricks automatically creates when a job runs and then terminates immediately after the job finishes.
        - Created on job start
        - Used exclusively for that job run
        - Automatically destroyed when done
    - In this case you pay for exactly the time your job runs -- no idle time.
    - **Key characteristics :**

    | Feature             | Description                                            |
    | ------------------- | ------------------------------------------------------ |
    | **Purpose**         | Run production jobs, ETL pipelines, or scheduled tasks |
    | **Lifecycle**       | Created → Job executes → Terminated automatically      |
    | **Attached Users**  | Only the job, not interactive users                    |
    | **Configuration**   | Defined as part of the job’s YAML or UI settings       |
    | **Cost Efficiency** | Very efficient — no idle cluster billing               |
    | **Isolation**       | Each job runs in its own clean environment             |
    | **Scalability**     | Can autoscale worker nodes dynamically                 |

    - **How do job clusters work internally**
        - You define a job in Databricks (Through UI, API or YAML)
        - Databricks provisions a new spark cluster
            - Uses your defined runtime
            - Installs libraries or wheels you specify
        - Executes the job tasks
        - Once finishes (success or failure), the cluster is terminated automatically.

- **Difference between Job cluster and All-purpose cluster**

    | Aspect         | All-Purpose Cluster             | Job Cluster                           |
    | -------------- | ------------------------------- | ------------------------------------- |
    | **Usage**      | Interactive dev & exploration   | Automated jobs / production pipelines |
    | **Lifecycle**  | Manually started and stopped    | Auto-created and auto-terminated      |
    | **Multi-user** | Supports multiple users         | Dedicated to a single job             |
    | **Cost**       | Billed while running, even idle | Billed only during job runtime        |
    | **Best For**   | Development / debugging         | Production automation                 |
    | **Example**    | Testing your PySpark script     | Running daily Bronze→Silver→Gold ETL  |

## Databricks Runtime
- Databricks runtime is a versioned environment that defines the core spark engine , libraries and ML stack used by clusters
- It's basically the software stack that runs on top of the cluster's virtual machines.
- When you start a cluster (all-purpose or job), you choose a Databricks Runtime version — that version determines which Spark version, Python version, Delta Lake, ML libraries, and optimizations are preinstalled.
- If Databricks clusters are the hardware 🖥️,
then Databricks Runtime is the operating system + libraries ⚙️ that make your Spark and ML code work.
- **What’s Inside a Databricks Runtime?**
    - Each runtime version comes with:
        - A specific versio of apache spark
        - Delta Lake built-in
        - Performance optimization by databricks (Like Photon, I/O caching)
        - Preinstalled Python/R/Scala/SQL libraries
        - Cluster-wide configuration and security patches
        - Integration with DBFS, Unity catalog, MLFlow etc.
- **Types of Databricks Runtimes**

    | Runtime Type                                             | Description                                                                | Use Case                                          |
    | -------------------------------------------------------- | -------------------------------------------------------------------------- | ------------------------------------------------- |
    | **Databricks Runtime (Standard)**                        | Core Spark + Delta Lake environment                                        | General data engineering, ETL, batch processing   |
    | **Databricks Runtime for Machine Learning (ML Runtime)** | Includes ML/AI libraries (scikit-learn, TensorFlow, XGBoost, MLflow, etc.) | Training, feature engineering, model deployment   |
    | **Databricks Runtime for Genomics**                      | Includes specialized libraries for genomics data processing                | Bioinformatics                                    |
    | **Databricks Runtime with Photon**                       | Uses Databricks’ high-performance *Photon* engine                          | High-speed data warehousing and analytics         |
    | **Databricks Light (Legacy)**                            | Lightweight version for basic jobs                                         | Legacy minimal environments (not recommended now) |


- Example Versions and Their Features :

    | Databricks Runtime | Apache Spark | Delta Lake | Python | Notes                                                  |
    | ------------------ | ------------ | ---------- | ------ | ------------------------------------------------------ |
    | **14.3 LTS**       | 3.5.x        | 3.x        | 3.10   | Stable, long-term support (recommended for production) |
    | **15.x**           | 3.5+         | 3.x        | 3.11   | Latest features, not LTS yet                           |
    | **13.3 LTS**       | 3.4.x        | 2.x        | 3.9    | Older but stable                                       |
    | **14.3 ML**        | 3.5.x        | 3.x        | 3.10   | Same as 14.3 but includes ML libraries                 |


- Typical Usage by Environment

    | Environment     | Recommended Runtime           | Reason                        |
    | --------------- | ----------------------------- | ----------------------------- |
    | **Development** | Latest stable version         | Access to newest features     |
    | **QA**          | Same as Prod (e.g., 14.3 LTS) | Ensures consistency           |
    | **Production**  | LTS runtime                   | Stability + long-term support |

## Cluster Visibility 
- Cluster visibility determines which users can see, attach to, and use a Databricks cluster in the workspace UI (e.g., when they open a notebook and choose a cluster to run on).
- **Why It Exists**
    - In small teams, everyone might share a few clusters.
    - But in enterprise setups, you often have:
        - Multiple departments
        - Different environments (dev/QA/prod)
        - Sensitive or costly compute resources
    - So Databricks lets you control visibility and permissions on clusters to avoid:
        - Unintended usage of production clusters
        - Unnecessary costs
        - Security or compliance violations (users seeing data they shouldn’t)
- Cluster Visibility vs Cluster Permissions

    | Concept                 | Purpose                                                                   |
    | ----------------------- | ------------------------------------------------------------------------- |
    | **Cluster Permissions** | Who can **start/stop**, **restart**, **edit**, or **attach** to a cluster |
    | **Cluster Visibility**  | Who can even **see** the cluster in the workspace UI                      |


- Where You Configure Visibility
    - In the UI (When Creating a Cluster)
        - "Access Mode" : Choose Single user, Shared or High Concurrency
        - Cluster permissions : Assign users/groups with "Can Attach", "Can Restart", etc...

    - In API/YAML
        - example : 
            ```python
            new_cluster:
            spark_version: 14.3.x-scala2.12
            node_type_id: Standard_DS3_v2
            num_workers: 2
            access_mode: SINGLE_USER  # or SHARED / HIGH_CONCURRENCY
            single_user_name: "aditya.kumar@databricks.com"
            ```

## Restarting and terminating clusters : 
Clusters can be restarted or terminated based on wheather you need to reset the environment or stop usage.
- **Restarting a cluster:**
    - Resets the environment without deleting the cluster.
        - Clears cached variables and memory
        - Fixes broken libraries or runtime errors
        - Keeps cluster config intact, reboots driver and workers
- **Terminating a cluster**
    - Fully stops the cluster to free resources and stop billing
        - Ends all active sessions and computations
        - Used when work is completed or cluster is idle
        - Must restart or recreate to use again.

## Jobs : 
- Databricks jobs are used to automate and schedule workloads like notebooks, workflows or scripts.
- It supports tasks orchestration with dependencies.
- Can run on jobs or all-purpose clusters.
- Triggered manually or on a schedule (CRON) or via API
- Supports retry plicies and email on failure.
- Tracks job runs , statuses and execution history.


## How to integrate github with databricks
https://youtu.be/0Hd5vYqin7w?t=8224 


## GIT 
- git is a distributed version control system (DVCS) created by linus torvold
- Each changes of your code (git commit) can be captured and tracked throughout the history of your project (git tree)
- **Common git commands:**
    - Repository : Represents logical container holding the codebase
    - Commit : Represents a change of data in the local repository
    - Tree : Represent the entire history of a repo
    - Remote : A version of your project hosted elsewhere, used for exchanging commits.
    - Branches : Divergent paths of development allowing isolated changes.
        - Main (Formally known as master) the most common name for the default branch
    - Clone : Creates a complete local copy of a repository, including its history.
    - Checkout : Switches between different branches or commits in your repo.
    - Pull : Downloads changes from a remote repository and merges then into your branch.
    - Push : This uploads your local repository changes to a remote repository.
    - Fetch : Downloads data from a remote repo without integrating it into your work.
    - Merge : This combines multiple commit history into one.
    - Staging files : Prepares and organizes changes for a commit.
        - Commit : Saves your changes as a snapshot in the local repository.
        - Add : Adds charges to the staging area for the next commit.

## Databricks vs Native Notebook versioning
| Feature / Aspect                          | **Databricks Notebook Versioning**                                   | **Native Git Versioning (Git-integrated notebooks)**               |
| ----------------------------------------- | -------------------------------------------------------------------- | ------------------------------------------------------------------ |
| **Where versions are stored**             | In Databricks workspace (proprietary)                                | In your Git repo (GitHub, GitLab, Bitbucket, Azure DevOps)         |
| **Versioning mechanism**                  | Automatic snapshots                                                  | Manual commits via Git                                             |
| **When versions are created**             | On every notebook save / edit                                        | Only when you `git add`, `commit`, and `push`                      |
| **Granularity**                           | Very fine-grained (every change)                                     | Commit-level (developer-controlled)                                |
| **Diff format**                           | Visual diff in Databricks UI; cell-level                             | Git diff; text-based or Markdown-formatted                         |
| **Revert options**                        | One-click "Restore Revision"                                         | Git revert/reset/checkout                                          |
| **Collaboration style**                   | Multiple users can edit the same notebook simultaneously (real-time) | Git handles merge conflicts; cannot edit notebook simultaneously   |
| **Conflict resolution**                   | Minimal conflicts due to UI locking                                  | Git merge conflicts possible (JSON structured notebooks)           |
| **Audit trail**                           | Author + timestamp stored in Databricks                              | Full Git history with messages, branches, PRs                      |
| **CI/CD suitability**                     | Weak — not integrated into pipelines                                 | Strong — native source control enables CI/CD                       |
| **Branching support**                     | No real branching                                                    | Full Git branching (dev, qa, prod workflows)                       |
| **Integration with IDEs (VS Code)**       | No                                                                   | Full support                                                       |
| **Use cases**                             | Quick iteration; notebook experimentation                            | Production-grade workflows                                         |
| **Reproducibility**                       | Low — hidden metadata + UI changes                                   | High — explicit code versioning                                    |
| **Exportability**                         | Export manually (.dbc or .ipynb)                                     | Already in repo — automatically portable                           |
| **Traceability**                          | Limited metadata                                                     | Full commit messages + PR reviews                                  |
| **Protection from accidental overwrites** | Medium                                                               | High — Git prevents force pushes unless allowed                    |
| **Best suited for**                       | Prototyping, exploration, data analysis                              | Software engineering, ETL pipelines, collaborative production code |


## Delta Lake
Delta Lake brings ACID transformations, schema control and performance improvements to data lakes.
- ACID transactions ensure reliability and consistency.
- Schema enforcement blocks bad data from being written.
- Schema evolution allows structure changes without breaking pipelines.
- Time travel lets you query old versions of data.
- Supports key commands like MERGE, DELETE, UPDATE and optimizations like Z-ORDER, OPTIMIZE and VACCUME.
## Delta Lake Time Travel
Query a specific version
```sql
    SELECT * FROM table_name VERSION AS OF 3;
    SELECT * FROM table_name TIMESTAMP AS OF '2025-11-14T09:32:45Z';
```
Restore table to earlier version
```sql
    RESTORE TABLE table_name TO VERSION AS OF 3;
    RESTORE TABLE table_name TO TIMESTAMP AS OF '2025-11-14T09:32:45Z';
```
- Time travel relies on _delta_log transaction history
- default retention is 30 days
- Great for when you have accidental DELETE, UPDATE or OVERWRITE

## Optimization
Databricks provides built-in tools to optimize delat performace and cleanup.
- **OPTIMIZE** compacts small files into ~1GB chunks (default size, configurable).
    - Delta lake stores data as many small paraquet files. Over time , because of streaming/incremental writes, updates, deletes etc. your table gets : 
        - many small files
        - files distributed unevenly across partitions
        - poor data skipping
        - slow query performance
        - slow Z-ORDER performance
        - slower MERGE operations
    - OPTIMIZE fixes this:
        - It compacts small files : produces large files (1GB default)
        - It optionally applies Z-ORDER : re-organizes data to improve query skipping
        - This results in : 
            - faster queries
            - faster MERGEs
            - reduced metadata load
            - better cluster utilization
    - **Why run optimize on a separate job cluster?**
        - A job cluster starts only for that job, then shuts down
        - This is the best practice because :
            - OPTIMIZE is cpu-heavy and memory-heavy
                - It reads lots of data , sorts it, compresses it, writes new paraquet files.
                - Running it on your main cluster can:
                    - Slow down production workloads
                    - Interfere with streaming jobs
                    - spike costs on your interactive cluster
        - Keeps your interactive cluster fast
            - Your user notebooks should not slow down because OPTIMIZE is consuming all compute.
        - Easier cost control
            - You can:
                - Choose cheaper/optimized instance types
                - run OPTIMIZE during off-peak hours (night)
                - shut down the cluster after job completes
    - **Why Use Compute-Optimized Instance Types?**
        - Compute optimized VMs (example: c5, c7i, c10 Standard_F series):
            - Faster CPUs
            - more cores
            - cheaper per-core cost
            - best for file compaction and sorting
            - faster ZORDER performance
        - Storage-optimized or memory optimized is not needed because:
            - OPTIMIZE is CPU + I/O bound not memeory heavy
            - ZORDER sorting benefits from more cores not more RAM
            - So compute-optimized = best performance at lowers cost.
    - **Why Run OPTIMIZE Regularly (Daily / Weekly)?**
        - To avoid the "small- files problem"
            - If your table is written frequently (streaming, CDC, MERGE), it accumulates thousands of small paraquet files.
            - Small files --> Slow queries --> high consts --> poor healthy.
            - Running OPTIMIZE regularly keeps file size healthy
        - To maintain ZORDER effectiveness : Z-ORDER improves skipping only if:
            - data is compacted
            - files are well-organized
            - As new data arrives, the benefits decreases unless OPTIMIZE is re-run.
        - To keep metadata small
            - Too many files = large Delta transaction logs.
            - OPTIMIZE reduces file count --> keeps metadata clean.
        - Better cluster performance 
            - When table files become balanced, Delta engines:
                - Plan faster
                - Read fewer files
                - push down filters better
        - Prevent performance degradation over time
            - If you don't run it for weeks/months :
                - performance slowly degrades
                - query latency increases
                - costs increases due to scanning more files.
            - Regular OPTIMIZE = stable, predictable performance.
    - OPTIMZE best practice:
        - Run OPTIMIZE on a separate job cluster
        - Use compute-optimized instance types
        - Schedule OPTIMIZE regularly (daily,weekly) for effitcient file layout
- **Detailed explaination : CPU-bound vs Memory-heavy vs memory-bound**
    - Memeory-heavy
        - This means:
            - OPTIMIZE uses substantial memory while preprocessing (because Spark does shuffles, caching, sorting etc.)
            - This is true
    - Memory-bound
        - This means:
            - OPTIMIZE performance is limited by memory and adding more memory makes it faster.
            - This is not true
            - OPTIMIZE does not get much faster if you increase memory from say 64GB to 256GB
            - Because the real bottlenecks are:
                - CPU for sorting 
                - DISK I/O for reading/writing Parquet files
                - Network shuffle
                - Those create the real performance cieling.
    - CPU-bound
        - This is true
        - Sorting + ZORDER are CPU-intensive operations. More vCPUs = faster OPTIMIZE.
    - I/O-bound
        - This is true.
        - Since OPTIMIZE re-writes the entire file layout:
            - reads thousands of paraquet files
            - writes them back optimized
            - Fast I/O (SSD-backed storage) and more worker nodes matter.
- **Why not run OPTIMIZE on your main cluster?**
    - Because even if you run OPTIMIZE , it is not memory-bound, it still uses memory and CPU to affect other workloads.
    - Example:
        - Your main cluster is running : 
            - Streaming jobs 
            - Dashboards
            - Data scientists experimenting 
            - interactive notebooks
        - If OPTIMIZE starts sorting billions of rows:
            - Spark reserves cores - others pause
            - memory is allocated for shuffles ---> other jobs spill
            - IO is slammed --> dashboard latency spikes
        - So it's simply best practice to isolate the workload.
- **ZORDER BY** physically organize data by selected columns to boost query performance.
    - Example : 
        ```sql
            OPTIMIZE table_name [WHERE predicate] 
            [ZORDER BY (col1 [, col2, ...])]
        ```
    - Use high-cardinality columns (example column_id) for **Z-Ordering**
    - **What is a high cardinality column?**
        - Columns that has many distinct / unique values are called high cardinality columns.
        - Columns that has few distinct values in the columns are called low-cardinality columns.
    - **Why Z-ORDER uses high-cardinality columns?**
        - Because Z-ORDER physically clusters data on disk so that rows with similar values end up close togeather.
        - Hence we can say that high cardinality means huge performance improvement.
        - **Why is this good?** : 
            - patient_id uniquely identifies a row.
            - queries like ```WHERE patient_id = 'P_1234'``` can skip almost all the files
        - **Why low-cardinality columns are bad for Z-order?**
            - But gender only has 3 values:
                - Male
                - Female
                - Other
            - Databricks can't meaningfully cluster data because.
                - too many rows share the same value
                - no string locality benefit
                - files skipping hardly improves
                - This wastes compute and money.
    - **Where Z_ORDER must be used?**
        - When your queries frequently filter on those columns.
        - When your data is large (100M+ rows)
        - When those columns have many distinct values
        ```sql
            SELECT * FROM diebetes.patients WHERE patient_id = `P1234`;
            SELECT * FROM diebetes.patients WHERE diagnosis_date BETWEEN '2024-01-01' AND '2024-12-31';
        ```
    - **Z-ORDER best practices** : 
        - Use high cardinality column for z-ordering
        - Avoid low-cardinality column
        - Limit ZORDER BY to 4 column or fewer
## What is a partition column?
- Parition = splitting a large table into separate folders based on column.
```bash
    /diabetes_data/
    month=2024-01/
    month=2024-02/
    month=2024-03/
```
- When you run : 
```sql
    WHERE month = '2024-03'
```
- Databricks reads only that folder, not the whole dataset - bog speed boost.
- **Why date columns commonly used for partitioning?**
    - Natural distribution
        - Most real-world data arrives daily.
            - daily logs
            - daily patient readings
            - daily transactions
            - hourly IOT data
        - So dates makes a perfect partition because data naturally fits into it.
    - Predictable query patterns
        - Most queries access data by recent time:
            - "Last 30 days"
            - "this week"
            - "last quater"
        - Partitioning by date makes these queries extremely fast.
    - Avoids small file problem
        - Data arrives in bacthes (daily/hourly)
        - Partitioning by date = each parition has enough data -> No Fragmentation.
    - Works well with Delta OPTIMIZE
        - Delta Lake compacting (OPTIMIZE Table) works best when partitions are logical and sized well.
- **Why not partition by high-cardinality columns?**
    - Example : ```patient_id```
    If you have partition by patient_id and you have 1Million patients then you get 1Million folders.
    - This leads to : 
        - Too many files
        - Too many metadata operations
        - Extreme slowdown
    - Hence the best practice is 
        - Partition by low to medium cardinality columns
        - Z-ORDER by high cardinality columns

### Why DELETE/UPDATE/MERGE create small files?
Delta Lake uses copy on write.

This means:

When you DELETE / UPDATE / MERGE create small files
- Delta does not modify existing parquet files directly. 
- Instead it:
    - Reads the old files that contain affected rows.
    - Applies DELETE/UPDATE/MERGE logic.
    - Writes new small Parquet files with only the change/remaining rows.
    - Marks old files as removed in the Delta log.
    - This results in lots of small files
        - Because:
            - Only the affected data is rewritten
            - The rewritten data often becomes fragmented
            - Many small 10KB ~ 20MB files get created

### Why small files are a problem
- Small fiels cause : 
    - More files opens
    - Spark scanning MANY tiny files
    - Slow queries
    - Slow streaming
    - More metadata overhead
- Hence we need **OPTIMZE** to compact them.

### What happens inside OPTIMIZE under the hood
- You can think of ```OPTIMIZE``` like "compaction + sorting"
- **UNDER THE HOOD:**
    - **Step 1: File Listing**
        - Delta scans the table for:
            - Small files
            - Files matching WHERE predicate (Optional)
        - What deos it means to run this query ```Files matching WHERE predicate (Optional)```
            - Example : 
            ```sql
            OPTIMIZE my_table
            WHERE event_date >= '2025-01-01'
            ZORDER BY (user_id)
            ```
            OR
            ```sql
            OPTIMIZE sales_table
            WHERE country = 'IN'
            ```
            - This WHERE clause is optional and it changes what OPTIMZE will do.
            - The WHERE predicate filters which Delta Lake data files are compacted
            - A delta table is made up of many small paraquet files:
            ```bash
                file1.parquet
                file2.parquet
                file3.parquet
                ...
            ```
            - When you run :
            ```sql
                OPTIMIZE my_table
            ```
            - Databricks will scan and compact ALL files in the table.
            - But when you run : 
            ```sql
                OPTIMIZE my_table WHERE event_table = '2025-01-01'
            ```
            - Databricks will only process files that contain rows where event_data = 2025-01-01
            - Files with other dates will not be touched, compacted or processed
            - Why is this important?
                - It saves cost
                    - OPTIMIZE reads and writes large amounts of data.
                    - If your table is partitioned by date (common practice):
                    ```bash
                    /event_date=2025-02-01/*.parquet
                    /event_date=2025-02-02/*.parquet
                    /event_date=2025-02-03/*.parquet
                    ```
                    - You can compact only the latest parition:
                    ```sql
                        OPTIMIZE events WHERE event_date >= current_date() - 7
                    ```
                    - Much cheaper 
                    - Much faster
                - Faster OPTIMIZE
                    - Instead of compacting the entire lake (maybe terabytes), you compact only hot data — recent partitions that have many small files.
                - Avoids uncessary reqwites
                    - Older partitions (say 2022) never change.
                    - They don't need OPTIMZE again.
                    - So : 
                    ```sql
                    OPTIMZE events WHERE event_date > date_sub(current_date(),30)
                    ```
                    These types of implementation is common.
    - **Step 2 : Read all data**
        - Spark reads all these small parquet files.
    - **Spark 3 : Shuffle + repartition**
        - Data is:
            - Shuffle (distributed across executors)
            - Repartitioned into fewer large paritions
            - Default target file size = 1GB
    - **Step 4 (Optional): ZORDER**
        - If you run : 
            ```sql
                OPTIMIZE table ZORDER BY (col1,col2)
            ``` 
        - Spark : 
            - Extracts Z-order curve values
            - Sorts records accordingly
            - Writes files grouped by similar values ---> improves skipping
    - **Step 5 : Wtire New compact fiels**
         -Spark writes large - 1GB files
    - **Step 6 : Delta Transaction Commit**
        - Delta Log:
            - Adds new files.
            - Removes small old files.
### How Databricks Photon improves OPTIMIZE performance
- Photon is:
    - A vectorized execution engine
    - A written C++
    - Optimized for parquet + Delta
- Photon boosts OPTIMIZE by : 
    - Faster Parquet decoding
        - Photon reads Parquet 2-3x faster than JVM spark.
    - Faster aggregation + sorting 
        - ZORDER requires large-scale sort --> Photon does this faster.
    - Better CPU untilization
        - Photon uses:
            - SMID instructions
            - Larger batches
            - Lower overhead per row
    - Less Garbage Collection pressure
        - Since photon bypasses the JVM for query execution, garbage collection is reduced.
    - Net impact : 
        - OPTIMIZE + Photon often becomes 2-5x faster
        - 20% ~ 40% cheaper
        - With improved file layout skipping 
### **Best OPTIMIZE scheduling strategy (Bronze / Silver / Gold)**
#### Bronze Layer (Raw data)
Characteristics : 
- Append-only (streaming ingestion)
- Many small batches 
- Lots of small files

**Best Strategy for Bronze :**
```bash
    RUN OPTIMIZE DAILY
```
Why?
- Streaming writes create TONS of small files
- High ingestion raate = high fragmentation
- Better compaction improves downstream performance.

#### Silver Layer (Cleaned + Enriched Data)
Characteristics : 
- Updates possible
- Merge operations common
- Joins and transformations create fragmented files

**Best Strategy for silver:**
```sql
    RUN OPTIMIZE EVERY 1–3 DAYS
```
Reason:
- MERGE / UPDATE / DELETE create small files.
- Too many small files harm analytical joins. 
- Regular compaction keeps silver smooth.

#### Gold Layer (Aggregated Data)
Characteristics:
- Small 
- Summaries
- Rarely updated

Best Strategy for Gold:
```bash
OPTIONAL — maybe once a week or month
```
Reason:
- Gold tables are small
- Heavy OPTIMIZE is unnecessary most of the time.

### What is Fragmentation in Delta Lake?
In Delta Lake (Databricks), fragmentation means: <br>
**Too many small files instead of fewer larger files.** <br>
Delat Lake works best when data is stored in fewer large paraquet files (~256MB to 1GB each) <br>
But many opertions create lots of tiny files causes problems.
#### Why does fragmentation happen?
If you ingest data many times per hour or continously (streaming), each write produces new small files.
| Ingestion Pattern            | Files created                             |
| ---------------------------- | ----------------------------------------- |
| Batch load 100 GB once       | 10 files of 1GB (good)                    |
| Streaming 1000 micro-batches | 1000 files of ~100MB (fragmented)         |
| Many small JSON/CSV sources  | Thousands of tiny files (very fragmented) |
#### DELETE / UPDATE / MERGE cause fragmentation
Because they don't re-write whole tables.
Internally they:
- Identify affected files
- Re-write only part of data
- Produce many new files
- Old files get "tombstoned" but still counted until vaccum
- This leads to a mix of lots of small leftover files + many new ones.
#### Why fragmentation is a problem?
It slows down queries because:
- Spark must lists thousands of files
- Spark must open thousands of small paraquet files
- File metadata grows huge 
- Caching becomes inefficient

As a result ?you end up paying
- more compute
- more memory
- more time

For the same query.